# Aula Prática U2 A3: Soluções em Inteligência Artificial (sistema fuzzy)

Fundamentos da Inteligência Artificial. Solução digital: Google Colab. Notebook: Avaliação de Qualidade de Atendimento.
Aluno: Roni Chittoni.

**Parte A**: as seis células do roteiro, reproduzidas como entregues (precedidas de A0, registro do ambiente).
**Parte B**: análises adicionais que as questões do roteiro pedem e o código original não produz: cobertura das partições, sobreposição zero, limite fixo versus fuzzy, pertinências para clientes menos tolerantes e uma nova regra.

| Evidência | Célula | O que capturar | Uso no relatório |
|---|---|---|---|
| E01 | A0 | versões e sistema | Materiais e Métodos |
| E02 | Célula 5 | saída dos quatro casos de teste | Resultados, Questões 3 e 4 |
| E03 | Célula 6 | funções de pertinência e superfície de qualidade | Questões 1 e 5 |
| E04 | B1 | tabelas de pertinência e cobertura; varredura do Caso 1 | Questões 1 e 4 |
| E05 | B2 | tabela e superfícies com e sem sobreposição | Questões 1 e 5 |
| E06 | B3 | limite fixo versus fuzzy em torno de 20 min | Questão 6 |
| E07 | B4 | pertinências para clientes menos tolerantes | Questão 7 |
| E08 | B5 | nova regra e superfície comparada | Questão 7 |


## Parte A: código do roteiro

In [ ]:
# A0: registro do ambiente de execução
import sys, platform, numpy, matplotlib
print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Sistema:", platform.platform())

In [ ]:
# Célula 1: Importar bibliotecas
import numpy as np
import matplotlib.pyplot as plt

# Configurar gráficos
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Célula 2: Definir funções de pertinência para Tempo de Espera
def tempo_rapido(x):
    """Função de pertinência para tempo rápido (0-15 min)"""
    if x <= 10:
        return 1.0
    elif x <= 20:
        return (20 - x) / 10
    else:
        return 0.0

def tempo_medio(x):
    """Função de pertinência para tempo médio (10-30 min)"""
    if x <= 10:
        return 0.0
    elif x <= 20:
        return (x - 10) / 10
    elif x <= 30:
        return (30 - x) / 10
    else:
        return 0.0

def tempo_lento(x):
    """Função de pertinência para tempo lento (25-60 min)"""
    if x <= 25:
        return 0.0
    elif x <= 35:
        return (x - 25) / 10
    else:
        return 1.0

In [ ]:
# Célula 3: Definir funções de pertinência para Cordialidade
def cordialidade_baixa(x):
    """Cordialidade baixa (0-4)"""
    if x <= 3:
        return 1.0
    elif x <= 5:
        return (5 - x) / 2
    else:
        return 0.0

def cordialidade_media(x):
    """Cordialidade média (3-7)"""
    if x <= 3:
        return 0.0
    elif x <= 5:
        return (x - 3) / 2
    elif x <= 7:
        return (7 - x) / 2
    else:
        return 0.0

def cordialidade_alta(x):
    """Cordialidade alta (6-10)"""
    if x <= 6:
        return 0.0
    elif x <= 8:
        return (x - 6) / 2
    else:
        return 1.0

In [ ]:
# Célula 4: Definir regras do sistema fuzzy
def avaliar_atendimento(tempo, cordialidade):
    """
    Regras:
    1. Se tempo é rápido E cordialidade é alta → Qualidade EXCELENTE
    2. Se tempo é rápido E cordialidade é média → Qualidade BOA
    3. Se tempo é médio E cordialidade é alta → Qualidade BOA
    4. Se tempo é médio E cordialidade é média → Qualidade REGULAR
    5. Caso contrário → Qualidade RUIM
    """

    # Calcular graus de pertinência
    t_rapido = tempo_rapido(tempo)
    t_medio = tempo_medio(tempo)
    t_lento = tempo_lento(tempo)

    c_baixa = cordialidade_baixa(cordialidade)
    c_media = cordialidade_media(cordialidade)
    c_alta = cordialidade_alta(cordialidade)

    # Aplicar regras (usando operador MIN para E)
    excelente = min(t_rapido, c_alta)
    boa = max(min(t_rapido, c_media), min(t_medio, c_alta))
    regular = min(t_medio, c_media)
    ruim = max(min(t_lento, c_baixa), min(t_lento, c_media), min(t_lento, c_alta),
               min(t_rapido, c_baixa), min(t_medio, c_baixa))

    # Defuzzificação simples (valor numérico final)
    qualidade_final = (excelente * 9 + boa * 7 + regular * 5 + ruim * 2) / max(excelente + boa + regular + ruim, 0.001)

    return {
        'excelente': excelente,
        'boa': boa,
        'regular': regular,
        'ruim': ruim,
        'qualidade_final': qualidade_final
    }

In [ ]:
# Célula 5: Testar o sistema
def testar_sistema():
    casos_teste = [
        {"tempo": 8, "cordialidade": 9, "descricao": "Atendimento rápido e muito cordial"},
        {"tempo": 15, "cordialidade": 6, "descricao": "Atendimento médio"},
        {"tempo": 35, "cordialidade": 3, "descricao": "Atendimento demorado e pouco cordial"},
        {"tempo": 12, "cordialidade": 8, "descricao": "Bom atendimento geral"}
    ]

    print("=== TESTE DO SISTEMA DE LÓGICA NEBULOSA ===\n")

    for i, caso in enumerate(casos_teste, 1):
        resultado = avaliar_atendimento(caso["tempo"], caso["cordialidade"])

        print(f"Caso {i}: {caso['descricao']}")
        print(f"  Tempo de espera: {caso['tempo']} minutos")
        print(f"  Cordialidade: {caso['cordialidade']}/10")
        print(f"  Graus de pertinência:")
        print(f"    Excelente: {resultado['excelente']:.2f}")
        print(f"    Boa: {resultado['boa']:.2f}")
        print(f"    Regular: {resultado['regular']:.2f}")
        print(f"    Ruim: {resultado['ruim']:.2f}")
        print(f"  Qualidade final: {resultado['qualidade_final']:.2f}/10\n")

testar_sistema()

In [ ]:
# Célula 6: Visualizar as funções de pertinência
def plotar_funcoes_pertinencia():
    # Tempo
    tempo_range = np.linspace(0, 60, 100)
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.plot(tempo_range, [tempo_rapido(t) for t in tempo_range], 'g-', label='Rápido', linewidth=2)
    plt.plot(tempo_range, [tempo_medio(t) for t in tempo_range], 'y-', label='Médio', linewidth=2)
    plt.plot(tempo_range, [tempo_lento(t) for t in tempo_range], 'r-', label='Lento', linewidth=2)
    plt.title('Tempo de Espera')
    plt.xlabel('Minutos')
    plt.ylabel('Grau de Pertinência')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Cordialidade
    cord_range = np.linspace(0, 10, 100)
    plt.subplot(1, 3, 2)
    plt.plot(cord_range, [cordialidade_baixa(c) for c in cord_range], 'r-', label='Baixa', linewidth=2)
    plt.plot(cord_range, [cordialidade_media(c) for c in cord_range], 'y-', label='Média', linewidth=2)
    plt.plot(cord_range, [cordialidade_alta(c) for c in cord_range], 'g-', label='Alta', linewidth=2)
    plt.title('Cordialidade')
    plt.xlabel('Pontuação (0-10)')
    plt.ylabel('Grau de Pertinência')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Superfície de qualidade
    plt.subplot(1, 3, 3)
    T, C = np.meshgrid(np.linspace(0, 60, 30), np.linspace(0, 10, 30))
    Z = np.zeros_like(T)

    for i in range(T.shape[0]):
        for j in range(T.shape[1]):
            resultado = avaliar_atendimento(T[i,j], C[i,j])
            Z[i,j] = resultado['qualidade_final']

    contour = plt.contourf(T, C, Z, levels=20, cmap='RdYlGn')
    plt.colorbar(contour)
    plt.title('Superfície de Qualidade')
    plt.xlabel('Tempo (min)')
    plt.ylabel('Cordialidade')

    plt.tight_layout()
    plt.show()

plotar_funcoes_pertinencia()

## Parte B: análises adicionais

In [ ]:
# B1: graus de pertinência em pontos-chave, cobertura das partições e varredura do Caso 1
import numpy as np
print("Tempo (min) | rápido | médio | lento | soma")
for t in [0, 5, 10, 12, 15, 20, 22, 25, 28, 30, 32, 35, 40, 60]:
    r, m, l = tempo_rapido(t), tempo_medio(t), tempo_lento(t)
    print(f"{t:11d} | {r:6.2f} | {m:5.2f} | {l:5.2f} | {r + m + l:4.2f}")

print("\nCordialidade | baixa | média | alta | soma")
for c in [0, 2, 3, 4, 5, 6, 6.5, 7, 8, 9, 10]:
    b, me, a = cordialidade_baixa(c), cordialidade_media(c), cordialidade_alta(c)
    print(f"{c:12.1f} | {b:5.2f} | {me:5.2f} | {a:4.2f} | {b + me + a:4.2f}")

print("\nVarredura do Caso 1 (cordialidade 9): a partir de que tempo 'excelente' deixa de valer 1,0?")
print("Tempo (min) | rápido | excelente | boa | qualidade final")
for t in [8, 9, 10, 11, 12, 14, 16, 18, 20]:
    r = avaliar_atendimento(t, 9)
    print(f"{t:11d} | {tempo_rapido(t):6.2f} | {r['excelente']:9.2f} | {r['boa']:4.2f} | {r['qualidade_final']:15.2f}")

In [ ]:
# B2: o que acontece com sobreposição zero entre funções de pertinência adjacentes
def tempo_rapido_0(x):  return 1.0 if x <= 15 else 0.0
def tempo_medio_0(x):   return 1.0 if 15 < x <= 30 else 0.0
def tempo_lento_0(x):   return 1.0 if x > 30 else 0.0
def cord_baixa_0(x):    return 1.0 if x <= 4 else 0.0
def cord_media_0(x):    return 1.0 if 4 < x <= 7 else 0.0
def cord_alta_0(x):     return 1.0 if x > 7 else 0.0

def avaliar_generico(tempo, cordialidade, fr, fm, fl, cb, cm, ca, regra_extra=False):
    """Mesmas cinco regras do roteiro, com funções de pertinência intercambiáveis.
    regra_extra=True acrescenta a regra 6: se tempo é lento E cordialidade é alta -> REGULAR (em vez de RUIM)."""
    t_rapido, t_medio, t_lento = fr(tempo), fm(tempo), fl(tempo)
    c_baixa, c_media, c_alta = cb(cordialidade), cm(cordialidade), ca(cordialidade)
    excelente = min(t_rapido, c_alta)
    boa = max(min(t_rapido, c_media), min(t_medio, c_alta))
    if regra_extra:
        regular = max(min(t_medio, c_media), min(t_lento, c_alta))
        ruim = max(min(t_lento, c_baixa), min(t_lento, c_media),
                   min(t_rapido, c_baixa), min(t_medio, c_baixa))
    else:
        regular = min(t_medio, c_media)
        ruim = max(min(t_lento, c_baixa), min(t_lento, c_media), min(t_lento, c_alta),
                   min(t_rapido, c_baixa), min(t_medio, c_baixa))
    return (excelente * 9 + boa * 7 + regular * 5 + ruim * 2) / max(excelente + boa + regular + ruim, 0.001)

sem_sobreposicao = (tempo_rapido_0, tempo_medio_0, tempo_lento_0, cord_baixa_0, cord_media_0, cord_alta_0)
print("Cordialidade 8 fixa. Tempo (min) | fuzzy do roteiro | sem sobreposição")
for t in [10, 12, 14, 15, 15.1, 16, 18, 20, 25, 28, 30, 30.1, 35]:
    q_f = avaliar_atendimento(t, 8)['qualidade_final']
    q_0 = avaliar_generico(t, 8, *sem_sobreposicao)
    print(f"{t:27.1f} | {q_f:16.2f} | {q_0:16.2f}")

T, C = np.meshgrid(np.linspace(0, 60, 121), np.linspace(0, 10, 101))
Z_roteiro = np.zeros_like(T); Z_zero = np.zeros_like(T)
for i in range(T.shape[0]):
    for j in range(T.shape[1]):
        Z_roteiro[i, j] = avaliar_atendimento(T[i, j], C[i, j])['qualidade_final']
        Z_zero[i, j] = avaliar_generico(T[i, j], C[i, j], *sem_sobreposicao)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, Z, titulo in zip(axes, [Z_roteiro, Z_zero],
                         ['Pertinências do roteiro (com sobreposição)', 'Pertinências sem sobreposição (limites rígidos)']):
    cf = ax.contourf(T, C, Z, levels=np.linspace(0, 10, 21), cmap='RdYlGn')
    ax.set_title(titulo); ax.set_xlabel('Tempo (min)'); ax.set_ylabel('Cordialidade')
fig.colorbar(cf, ax=axes.ravel().tolist(), label='Qualidade final')
plt.show()

In [ ]:
# B3: classificador binário com limite fixo versus sistema fuzzy em torno de 20 minutos
def classificador_binario(tempo, cordialidade, limite_tempo=20, limite_cord=6):
    return "bom" if (tempo <= limite_tempo and cordialidade >= limite_cord) else "ruim"

print("Tempo (min) | Cord. | Binário (<=20 min e >=6) | Fuzzy (0-10) | rápido/médio/lento")
for t, c in [(19.0, 8), (19.9, 8), (20.0, 8), (20.1, 8), (21.0, 8), (25.0, 8), (28.0, 8), (30.0, 8), (35.0, 8), (20.0, 5.9), (20.0, 6.0)]:
    r = avaliar_atendimento(t, c)
    print(f"{t:11.1f} | {c:5.1f} | {classificador_binario(t, c):>24} | {r['qualidade_final']:12.2f} | "
          f"{tempo_rapido(t):.2f}/{tempo_medio(t):.2f}/{tempo_lento(t):.2f}")

tempos = np.linspace(0, 60, 601)
fuzzy_c8 = [avaliar_atendimento(t, 8)['qualidade_final'] for t in tempos]
binario_c8 = [10.0 if classificador_binario(t, 8) == "bom" else 0.0 for t in tempos]
plt.figure(figsize=(10, 4.5))
plt.plot(tempos, fuzzy_c8, linewidth=2, label='Sistema fuzzy (qualidade 0-10)')
plt.plot(tempos, binario_c8, 'k--', linewidth=2, label='Limite fixo em 20 min (bom = 10, ruim = 0)')
plt.axvline(20, color='gray', linewidth=0.8)
plt.title('Qualidade em função do tempo de espera, cordialidade 8')
plt.xlabel('Tempo de espera (min)'); plt.ylabel('Qualidade'); plt.grid(True, alpha=0.3); plt.legend()
plt.show()

In [ ]:
# B4: funções de pertinência para clientes menos tolerantes ao tempo de espera
def tempo_rapido_v2(x):
    """Rápido: núcleo até 5 min, transição até 10 min"""
    if x <= 5: return 1.0
    elif x <= 10: return (10 - x) / 5
    else: return 0.0

def tempo_medio_v2(x):
    """Médio: sobe de 5 a 10 min, desce de 10 a 20 min"""
    if x <= 5: return 0.0
    elif x <= 10: return (x - 5) / 5
    elif x <= 20: return (20 - x) / 10
    else: return 0.0

def tempo_lento_v2(x):
    """Lento: sobe de 15 a 25 min, saturado a partir de 25 min"""
    if x <= 15: return 0.0
    elif x <= 25: return (x - 15) / 10
    else: return 1.0

menos_tolerantes = (tempo_rapido_v2, tempo_medio_v2, tempo_lento_v2,
                    cordialidade_baixa, cordialidade_media, cordialidade_alta)
print("Caso | tempo | cord. | qualidade (roteiro) | qualidade (menos tolerantes)")
for i, (t, c) in enumerate([(8, 9), (15, 6), (35, 3), (12, 8)], 1):
    q1 = avaliar_atendimento(t, c)['qualidade_final']
    q2 = avaliar_generico(t, c, *menos_tolerantes)
    print(f"{i:4d} | {t:5d} | {c:5d} | {q1:19.2f} | {q2:28.2f}")

tr = np.linspace(0, 60, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (fr, fm, fl), titulo in zip(axes, [(tempo_rapido, tempo_medio, tempo_lento), (tempo_rapido_v2, tempo_medio_v2, tempo_lento_v2)],
                                    ['Tempo de espera: pertinências do roteiro', 'Tempo de espera: clientes menos tolerantes']):
    ax.plot(tr, [fr(t) for t in tr], 'g-', label='Rápido', linewidth=2)
    ax.plot(tr, [fm(t) for t in tr], 'y-', label='Médio', linewidth=2)
    ax.plot(tr, [fl(t) for t in tr], 'r-', label='Lento', linewidth=2)
    ax.set_title(titulo); ax.set_xlabel('Minutos'); ax.set_ylabel('Grau de pertinência'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# B5: nova regra proposta: se tempo é lento E cordialidade é alta -> Qualidade REGULAR (em vez de RUIM)
originais = (tempo_rapido, tempo_medio, tempo_lento, cordialidade_baixa, cordialidade_media, cordialidade_alta)
print("Tempo | Cord. | 5 regras (roteiro) | 6 regras (nova regra)")
for t, c in [(8, 9), (15, 6), (35, 3), (12, 8), (28, 9), (30, 9), (40, 9), (40, 5), (40, 2)]:
    q5 = avaliar_generico(t, c, *originais)
    q6 = avaliar_generico(t, c, *originais, regra_extra=True)
    print(f"{t:5d} | {c:5d} | {q5:18.2f} | {q6:21.2f}")

Z_nova = np.zeros_like(T)
for i in range(T.shape[0]):
    for j in range(T.shape[1]):
        Z_nova[i, j] = avaliar_generico(T[i, j], C[i, j], *originais, regra_extra=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, Z, titulo in zip(axes, [Z_roteiro, Z_nova], ['Cinco regras do roteiro', 'Com a regra 6 (lento E alta -> regular)']):
    cf = ax.contourf(T, C, Z, levels=np.linspace(0, 10, 21), cmap='RdYlGn')
    ax.set_title(titulo); ax.set_xlabel('Tempo (min)'); ax.set_ylabel('Cordialidade')
fig.colorbar(cf, ax=axes.ravel().tolist(), label='Qualidade final')
plt.show()

## Entrega

Checklist do roteiro: Colab funcionando (A0), funções de pertinência (Células 2 e 3), sistema de regras (Célula 4), testes com diferentes valores (Célula 5 e Parte B), gráficos (Célula 6 e Parte B), análise registrada (relatório). Para o último item: Arquivo > Imprimir > Salvar como PDF, ou copiar os resultados para o relatório em Word.